In [1]:
import pandas as pd
import os

data_path = "../Race data from 1950 to 2026 Race 9"

print(os.listdir(data_path))

['seasons.csv', 'sprint_results.csv', 'driver_standings.csv', 'constructor_results.csv', 'pit_stops.csv', 'lap_times.csv', 'results.csv', 'races.csv', 'circuits.csv', 'constructors.csv', 'qualifying.csv', 'status.csv', 'drivers.csv', 'constructor_standings.csv']


In [2]:
import pandas as pd
import os

data_path = "../Race data from 1950 to 2026 Race 9"

# Tell pandas that '\N' means "missing" while loading
na_vals = ['\\N']

races = pd.read_csv(f"{data_path}/races.csv", na_values=na_vals)
results = pd.read_csv(f"{data_path}/results.csv", na_values=na_vals)
qualifying = pd.read_csv(f"{data_path}/qualifying.csv", na_values=na_vals)
drivers = pd.read_csv(f"{data_path}/drivers.csv", na_values=na_vals)
constructors = pd.read_csv(f"{data_path}/constructors.csv", na_values=na_vals)
driver_standings = pd.read_csv(f"{data_path}/driver_standings.csv", na_values=na_vals)
constructor_standings = pd.read_csv(f"{data_path}/constructor_standings.csv", na_values=na_vals)
status = pd.read_csv(f"{data_path}/status.csv", na_values=na_vals)

# Now position should properly show NaN for DNFs instead of the string '\N'
print(results['position'].isna().sum(), "DNFs / non-classified results out of", len(results))
print(results.dtypes[['grid', 'position']])

10953 DNFs / non-classified results out of 27436
grid        float64
position    float64
dtype: object


In [3]:
# Start with results as the base (one row per driver per race)
df = results.merge(races[['raceId', 'year', 'round', 'circuitId', 'name', 'date']], 
                    on='raceId', how='left', suffixes=('', '_race'))

df = df.merge(drivers[['driverId', 'driverRef', 'forename', 'surname', 'nationality']], 
              on='driverId', how='left', suffixes=('', '_driver'))

df = df.merge(constructors[['constructorId', 'constructorRef', 'name', 'nationality']], 
              on='constructorId', how='left', suffixes=('', '_constructor'))

df = df.merge(status, on='statusId', how='left')

print(df.shape)
df.head(3)

(27436, 31)


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,name,date,driverRef,forename,surname,nationality,constructorRef,name_constructor,nationality_constructor,status
0,1,18,1,1,22.0,1.0,1.0,1,1,10.0,...,Australian Grand Prix,2008-03-16,hamilton,Lewis,Hamilton,British,mclaren,McLaren,British,Finished
1,2,18,2,2,3.0,5.0,2.0,2,2,8.0,...,Australian Grand Prix,2008-03-16,heidfeld,Nick,Heidfeld,German,bmw_sauber,BMW Sauber,German,Finished
2,3,18,3,3,7.0,7.0,3.0,3,3,6.0,...,Australian Grand Prix,2008-03-16,rosberg,Nico,Rosberg,German,williams,Williams,British,Finished


In [4]:
print(df.columns.tolist())

['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId', 'year', 'round', 'circuitId', 'name', 'date', 'driverRef', 'forename', 'surname', 'nationality', 'constructorRef', 'name_constructor', 'nationality_constructor', 'status']


In [5]:
df = df.merge(
    qualifying[['raceId', 'driverId', 'position']].rename(columns={'position': 'quali_position'}),
    on=['raceId', 'driverId'],
    how='left'
)

print(df.shape)
print(df['quali_position'].isna().sum(), "rows missing qualifying data out of", len(df))

(27436, 32)
16269 rows missing qualifying data out of 27436


In [6]:
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)

Shape: (27436, 32)

Data types:
 resultId                     int64
raceId                       int64
driverId                     int64
constructorId                int64
number                     float64
grid                       float64
position                   float64
positionText                   str
positionOrder                int64
points                     float64
laps                         int64
time                           str
milliseconds               float64
fastestLap                 float64
rank                       float64
fastestLapTime                 str
fastestLapSpeed            float64
statusId                     int64
year                         int64
round                        int64
circuitId                    int64
name                           str
date                           str
driverRef                      str
forename                       str
surname                        str
nationality                    str
constructorRef        

In [7]:
#missing data per column
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

,missing_count,missing_pct
milliseconds,19326,70.4
time,19326,70.4
fastestLapSpeed,19184,69.9
fastestLapTime,18538,67.6
fastestLap,18538,67.6
rank,18280,66.6
quali_position,16269,59.3
position,10953,39.9
grid,20,0.1
number,6,0.0


In [8]:
#is missing quali_position concentrated in certain years
df[df['quali_position'].isna()]['year'].value_counts().sort_index()

year
1950    160
1951    179
1952    215
1953    246
1954    230
       ... 
2017      2
2019      2
2021      1
2025      1
2026      3
Name: count, Length: 62, dtype: int64

In [9]:
#years/rounds this dataset covers
print("Year range:", df['year'].min(), "-", df['year'].max())
print("\nRaces per year (sample):")
print(df.groupby('year')['raceId'].nunique().tail(10))

Year range: 1950 - 2026

Races per year (sample):
year
2017    20
2018    21
2019    21
2020    17
2021    22
2022    22
2023    22
2024    24
2025    24
2026     9
Name: raceId, dtype: int64


In [10]:
# Distribution of key numeric columns
df[['grid', 'positionOrder', 'points', 'laps']].describe()

,grid,positionOrder,points,laps
count,27416.000000,27436.000000,27436.000000,27436.000000
mean,11.126459,12.744314,2.060069,46.501677
std,7.175740,7.634701,4.463888,29.289082
min,0.000000,1.000000,0.000000,0.000000
25%,5.000000,6.000000,0.000000,24.000000
50%,11.000000,12.000000,0.000000,53.000000
75%,17.000000,18.000000,2.000000,66.000000
max,34.000000,39.000000,50.000000,200.000000


In [11]:
#how many drivers per race
df.groupby('raceId')['driverId'].nunique().describe()

count    1158.000000
mean       23.613990
std         4.698006
min        10.000000
25%        20.000000
50%        22.000000
75%        26.000000
max        42.000000
Name: driverId, dtype: float64

In [12]:
# Which races have unusually high driver counts?
race_counts = df.groupby('raceId')['driverId'].nunique().sort_values(ascending=False)
print(race_counts.head(10))

#what year/name those races are
big_races = race_counts.head(5).index
df[df['raceId'].isin(big_races)][['raceId', 'year', 'name']].drop_duplicates()

raceId
800    42
363    39
365    39
362    39
361    39
357    39
368    39
367    39
366    39
360    39
Name: driverId, dtype: int64


,raceId,year,name
8066,361,1989,Canadian Grand Prix
8105,362,1989,French Grand Prix
8144,363,1989,British Grand Prix
8221,365,1989,Hungarian Grand Prix
19231,800,1954,Indianapolis 500


In [13]:
pd.set_option('display.max_columns', None)  # show all columns, not truncated
df.sample(10)  # 10 random rows instead of just the first few — more representative

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,year,round,circuitId,name,date,driverRef,forename,surname,nationality,constructorRef,name_constructor,nationality_constructor,status,quali_position
14025,14026,571,231,1,11.0,2.0,1.0,1,1,9.0,75,1:44:52.09,6292090.0,NaN,NaN,NaN,NaN,1,1976,12,39,Dutch Grand Prix,1976-08-29,hunt,James,Hunt,British,mclaren,McLaren,British,Finished,NaN
5886,5887,284,110,25,4.0,17.0,NaN,R,19,0.0,24,NaN,NaN,NaN,NaN,NaN,NaN,5,1993,12,13,Belgian Grand Prix,1993-08-29,cesaris,Andrea,de Cesaris,Italian,tyrrell,Tyrrell,British,Engine,NaN
7087,7088,323,118,32,11.0,10.0,7.0,7,7,0.0,60,NaN,NaN,NaN,NaN,NaN,NaN,11,1990,3,21,San Marino Grand Prix,1990-05-13,warwick,Derek,Warwick,British,team_lotus,Team Lotus,British,+1 Lap,NaN
19898,19899,826,629,113,23.0,18.0,NaN,R,28,0.0,30,NaN,NaN,NaN,NaN,NaN,NaN,109,1951,2,19,Indianapolis 500,1951-05-30,griffith,Cliff,Griffith,American,kurtis_kraft,Kurtis Kraft,American,Axle,NaN
16036,16037,649,346,37,15.0,20.0,NaN,R,19,0.0,19,NaN,NaN,NaN,NaN,NaN,NaN,22,1970,7,38,British Grand Prix,1970-07-18,siffert,Jo,Siffert,Swiss,march,March,British,Suspension,NaN
1633,1634,95,32,19,15.0,14.0,NaN,R,20,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,3,2004,6,6,Monaco Grand Prix,2004-05-23,klien,Christian,Klien,Austrian,jaguar,Jaguar,British,Accident,14.0
6769,6770,313,78,40,34.0,24.0,NaN,R,26,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,20,1991,9,10,German Grand Prix,1991-07-28,larini,Nicola,Larini,Italian,lambo,Lambo,Italian,Spun off,NaN
17054,17055,702,289,66,10.0,13.0,5.0,5,5,2.0,39,NaN,NaN,NaN,NaN,NaN,NaN,11,1965,4,51,French Grand Prix,1965-06-27,hill,Graham,Hill,British,brm,BRM,British,+1 Lap,NaN
26129,26135,1113,825,210,20.0,6.0,10.0,10,10,1.0,62,+72.116,6469534.0,48.0,10.0,1:38.107,181.271,1,2023,15,15,Singapore Grand Prix,2023-09-17,kevin_magnussen,Kevin,Magnussen,Danish,haas,Haas F1 Team,American,Finished,6.0
135,136,24,20,5,15.0,19.0,8.0,8,8,1.0,70,+54.120,5838567.0,33.0,12.0,1:18.532,199.913,1,2008,7,7,Canadian Grand Prix,2008-06-08,vettel,Sebastian,Vettel,German,toro_rosso,Toro Rosso,Italian,Finished,20.0


In [14]:
df[df['points'] > 25][['year', 'name', 'points', 'positionOrder']].sort_values('points', ascending=False).head(10)

,year,name,points,positionOrder
22514,2014,Abu Dhabi Grand Prix,50.0,1
22515,2014,Abu Dhabi Grand Prix,36.0,2
22516,2014,Abu Dhabi Grand Prix,30.0,3
24197,2019,Australian Grand Prix,26.0,1
24277,2019,Spanish Grand Prix,26.0,1
24360,2019,Austrian Grand Prix,26.0,1
24380,2019,British Grand Prix,26.0,1
24400,2019,German Grand Prix,26.0,1
24500,2019,Russian Grand Prix,26.0,1
24601,2019,Abu Dhabi Grand Prix,26.0,1


In [15]:
df[df['grid'] == 0][['year', 'name', 'grid', 'positionOrder', 'points']].head(10)

,year,name,grid,positionOrder,points
2281,2002,San Marino Grand Prix,0.0,22,0.0
2432,2002,French Grand Prix,0.0,20,0.0
2433,2002,French Grand Prix,0.0,21,0.0
2434,2002,French Grand Prix,0.0,22,0.0
2797,2001,British Grand Prix,0.0,22,0.0
2972,2000,Brazilian Grand Prix,0.0,21,0.0
2973,2000,Brazilian Grand Prix,0.0,22,0.0
3346,1999,Brazilian Grand Prix,0.0,22,0.0
3785,1998,Monaco Grand Prix,0.0,22,0.0
4003,1998,Japanese Grand Prix,0.0,22,0.0


In [16]:
#recodepit lane start as one worse than the last real grid slot, per race
max_grid_per_race = df.groupby('raceId')['grid'].transform('max')
df['grid_fixed'] = df['grid'].replace(0, pd.NA)
df['grid_fixed'] = df['grid_fixed'].fillna(max_grid_per_race + 1)

# Sanity check
print(df[['grid', 'grid_fixed']].describe())

               grid
count  27416.000000
mean      11.126459
std        7.175740
min        0.000000
25%        5.000000
50%       11.000000
75%       17.000000
max       34.000000


In [17]:
import sys
print(sys.executable)

/home/mariam/f1-race-predictor/venv/bin/python


In [18]:
import sys
!{sys.executable} -m pip install pyarrow


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [19]:
import pandas as pd
import os
os.makedirs("../data_processed", exist_ok=True)
df.to_parquet("../data_processed/race_results_clean.parquet", index=False)
print("Saved:", df.shape)

Saved: (27436, 33)
